# 01 - Segments, and PII gone before anything is embedded

**Business question:** what are customers telling the sales and success teams on calls, and can we learn from it without
copying transcripts into another system or exposing customer PII?

**In this notebook** the call-recording data that already lands in Snowflake is cut into conversation segments,
redacted, and embedded, in that order, with each step processing only rows it has not seen.

| Section | What it shows |
|---|---|
| 1 | What landed: calls, turns, and how many are short dials that never need an AI call |
| 2 | A masking policy on the raw transcript column |
| 3 | Segments: windows of four consecutive turns, as an incremental dynamic table |
| 4 | `AI_REDACT` on each new segment, through a stream and a MERGE |
| 5 | `AI_EMBED` on redacted text only, as an incremental dynamic table |
| 6 | A new month lands: only the new rows are redacted and embedded |

Dependencies come from `./demo.sh setup` (no `%pip` cell). Every section ends in a `CHECK` that prints PASS or FAIL.

In [ ]:
NOTEBOOK_TAG = "transcript-churn-01"

# Session: Snowsight Workspaces supplies one; a local kernel (Cortex Code Desktop,
# ./demo.sh build) opens the connection saved in .env by ./demo.sh setup.
import json
import os
import time

import numpy as np
import pandas as pd

DB = "TRANSCRIPT_CHURN_ML_DEMO"
WH = "TRANSCRIPT_CHURN_ML_WH"
ROLE = "SYSADMIN"
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"
LLM = "claude-sonnet-4-5"
# The account this demo is allowed to build in. ./demo.sh setup writes it to .env;
# in Workspaces, replace the placeholder with your demo account name.
DEMO_ACCOUNT = os.environ.get("DEMO_EXPECTED_ACCOUNT", "<YOUR_DEMO_ACCOUNT>")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    LOCAL = False
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config(
        "connection_name", os.environ.get("SNOWFLAKE_DEFAULT_CONNECTION_NAME", "default")).create()
    LOCAL = True

session.use_role(ROLE)
session.use_warehouse(WH)
session.use_database(DB)
session.sql(f"ALTER SESSION SET QUERY_TAG = '{NOTEBOOK_TAG}'").collect()

if LOCAL:
    # Local kernels have no SQL cells: run each %%sql cell through this session instead.
    from IPython.core.magic import register_cell_magic
    from IPython.display import display

    @register_cell_magic
    def sql(line, cell):
        args = line.split()
        name = args[args.index("-r") + 1] if "-r" in args else None
        rows = session.sql(cell).collect()      # collect(): CALL and EXECUTE cannot be described first
        result = pd.DataFrame([r.as_dict() for r in rows])
        if name:
            get_ipython().user_ns[name] = result
        display(result.head(50))

CHECKS = {}


def check(section, ok, evidence):
    status = "PASS" if ok else "FAIL"
    CHECKS[section] = (status, evidence)
    print(f"CHECK {section}: {status} | {evidence}")
    if not ok:
        raise AssertionError(f"CHECK {section} failed: fix the cause and re-run this section")


def q(text):
    return session.sql(text).to_pandas()


def one(text):
    return session.sql(text).collect()[0][0]


account, role = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_ROLE()").collect()[0]
if account != DEMO_ACCOUNT:
    raise SystemExit(f"STOP: connected to {account}, but this demo may only build in {DEMO_ACCOUNT}. "
                     "Set DEMO_EXPECTED_ACCOUNT (./demo.sh setup does) or edit DEMO_ACCOUNT in this cell.")
if not session.sql(f"SHOW SCHEMAS LIKE 'RAW' IN DATABASE {DB}").collect():
    raise SystemExit(f"STOP: {DB}.RAW is missing. Run lab/setup.sql (./demo.sh setup) first.")
print(f"Connected as role {role} on {account}, warehouse {WH}, query tag {NOTEBOOK_TAG} "
      f"({'local kernel' if LOCAL else 'Workspaces session'})")

---
## 1. What landed

Two months of recorded calls (November and December) sit in `RAW.CALLS` and `RAW.UTTERANCES`, one row per speaker
turn. A third month is held back in `RAW.*_NEW_MONTH` and arrives in Section 6. This cell first resets the raw tables to
the two history months, so the notebook can be re-run.

In [ ]:
%%sql -r reset_history_turns
DELETE FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.UTTERANCES
WHERE CALL_ID IN (SELECT CALL_ID FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS_NEW_MONTH)

In [ ]:
%%sql -r reset_history_calls
DELETE FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS
WHERE CALL_ID IN (SELECT CALL_ID FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS_NEW_MONTH)

In [ ]:
%%sql -r landed_profile
SELECT CALL_TYPE, COUNT(*) AS CALLS, ROUND(AVG(DURATION_SEC) / 60, 1) AS AVG_MINUTES
FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS GROUP BY 1 ORDER BY 2 DESC

In [ ]:
calls = one(f"SELECT COUNT(*) FROM {DB}.RAW.CALLS")
dials = one(f"SELECT COUNT_IF(CALL_TYPE = 'OUTBOUND_DIAL') FROM {DB}.RAW.CALLS")
turns = one(f"SELECT COUNT(*) FROM {DB}.RAW.UTTERANCES")
latest = one(f"SELECT MAX(CALL_TS)::DATE::STRING FROM {DB}.RAW.CALLS")
DIAL_SHARE = dials / calls
check("S1", calls > 0 and 0.5 < DIAL_SHARE < 0.75 and latest < "2026-01-01",
      f"{calls:,} history calls ({DIAL_SHARE:.0%} short dials, no conversation), {turns:,} turns, latest call {latest}")

---
## 2. Govern the raw text first

The raw transcript column carries names, phone numbers and email addresses. A masking policy returns it only to the
pipeline owner role; anyone else querying `RAW.UTTERANCES` sees a placeholder. Analysts, models and LLM calls work from
the redacted segments built below, never from this column.

In [ ]:
%%sql -r create_masking_policy
CREATE MASKING POLICY IF NOT EXISTS TRANSCRIPT_CHURN_ML_DEMO.RAW.TRANSCRIPT_TEXT_MASK AS (VAL STRING) RETURNS STRING ->
  CASE WHEN IS_ROLE_IN_SESSION('SYSADMIN') THEN VAL ELSE '[masked: call transcript text]' END
  COMMENT = 'Raw transcript text is visible only to the pipeline owner role'

In [ ]:
%%sql -r attach_masking_policy
ALTER TABLE TRANSCRIPT_CHURN_ML_DEMO.RAW.UTTERANCES MODIFY COLUMN TEXT
  SET MASKING POLICY TRANSCRIPT_CHURN_ML_DEMO.RAW.TRANSCRIPT_TEXT_MASK FORCE

In [ ]:
refs = q(f"""SELECT POLICY_NAME, REF_ENTITY_NAME, REF_COLUMN_NAME FROM TABLE({DB}.INFORMATION_SCHEMA.POLICY_REFERENCES(
         REF_ENTITY_NAME => '{DB}.RAW.UTTERANCES', REF_ENTITY_DOMAIN => 'table'))""")
ok = ((refs["POLICY_NAME"] == "TRANSCRIPT_TEXT_MASK") & (refs["REF_COLUMN_NAME"] == "TEXT")).any()
check("S2", bool(ok), f"policy references on RAW.UTTERANCES: {refs.to_dict('records')}")

---
## 3. Segments: the unit of analysis

A whole call mixes many topics, and a single turn is too short to mean much. A **segment** is a window of four
consecutive turns (rep, customer, rep, customer), about 100-200 tokens, keeping the call type and the account. Short
dials are dropped here, before any AI function runs: that is the cheapest cost control there is.

`CORE.SEGMENTS` is a dynamic table with `REFRESH_MODE = INCREMENTAL`: when new turns land, only the affected calls are
recomputed.

In [ ]:
%%sql -r create_segments
CREATE OR REPLACE DYNAMIC TABLE TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS
  TARGET_LAG = 'DOWNSTREAM' WAREHOUSE = TRANSCRIPT_CHURN_ML_WH REFRESH_MODE = INCREMENTAL
  COMMENT = 'Windows of four consecutive turns per real conversation; short dials are filtered out before any AI cost'
AS
SELECT u.CALL_ID || '-' || FLOOR((u.TURN_NO - 1) / 4) AS SEGMENT_ID,
       u.CALL_ID, c.ACCOUNT_ID, c.CALL_TS, c.CALL_TYPE, c.OPPORTUNITY_ID,
       FLOOR((u.TURN_NO - 1) / 4) AS SEGMENT_NO,
       COUNT(*) AS N_TURNS,
       LISTAGG(u.SPEAKER_SIDE || ': ' || u.TEXT, '\n') WITHIN GROUP (ORDER BY u.TURN_NO) AS SEGMENT_TEXT,
       LISTAGG(IFF(u.SPEAKER_SIDE = 'CUSTOMER', u.TEXT, NULL), ' ') WITHIN GROUP (ORDER BY u.TURN_NO) AS CUSTOMER_TEXT
FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.UTTERANCES u
JOIN TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS c ON c.CALL_ID = u.CALL_ID
WHERE c.CALL_TYPE <> 'OUTBOUND_DIAL'
GROUP BY 1, 2, 3, 4, 5, 6, 7

In [ ]:
%%sql -r segment_tokens
-- AI_COUNT_TOKENS estimates the input tokens a full pass would bill, before running it
SELECT COUNT(*) AS SEGMENTS,
       ROUND(AVG(AI_COUNT_TOKENS('ai_embed', 'snowflake-arctic-embed-l-v2.0', SEGMENT_TEXT))) AS AVG_EMBED_TOKENS,
       MAX(AI_COUNT_TOKENS('ai_embed', 'snowflake-arctic-embed-l-v2.0', SEGMENT_TEXT)) AS MAX_EMBED_TOKENS,
       SUM(AI_COUNT_TOKENS('ai_embed', 'snowflake-arctic-embed-l-v2.0', SEGMENT_TEXT)) AS TOTAL_EMBED_TOKENS
FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS

In [ ]:
dt = session.sql(f"SHOW DYNAMIC TABLES LIKE 'SEGMENTS' IN SCHEMA {DB}.CORE").collect()[0]
tok = segment_tokens.iloc[0]
HIST_SEGMENTS = int(tok["SEGMENTS"])
EMBED_TOKENS_HISTORY = int(tok["TOTAL_EMBED_TOKENS"])
check("S3", dt["refresh_mode"] == "INCREMENTAL" and HIST_SEGMENTS > 0 and int(tok["MAX_EMBED_TOKENS"]) <= 512,
      f"CORE.SEGMENTS refresh_mode {dt['refresh_mode']}; {HIST_SEGMENTS:,} segments, avg {int(tok['AVG_EMBED_TOKENS'])} "
      f"tokens (max {int(tok['MAX_EMBED_TOKENS'])}); full embed pass = {EMBED_TOKENS_HISTORY:,} input tokens")

---
## 4. Redact before anything else touches the text

`AI_REDACT` replaces names, phone numbers, email addresses and other PII with placeholders such as `[NAME]`. A stream on
`CORE.SEGMENTS` remembers which segments are new, and `CORE.REDACT_NEW_SEGMENTS()` MERGEs only those rows, so a segment is
redacted (and billed) once. `AI_REDACT` accepts up to 4,096 input-plus-output tokens per call, which is why segmentation
comes first.

In [ ]:
%%sql -r create_redacted_table
CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_REDACTED (
  SEGMENT_ID STRING, CALL_ID STRING, ACCOUNT_ID STRING, CALL_TS TIMESTAMP_NTZ, CALL_TYPE STRING,
  OPPORTUNITY_ID STRING, SEGMENT_NO INT, REDACTED_TEXT STRING, REDACTED_CUSTOMER_TEXT STRING,
  KNOWN_NAME_FIXES INT, REDACTED_AT TIMESTAMP_NTZ)
  CHANGE_TRACKING = TRUE
  COMMENT = 'Segments after AI_REDACT plus a known-name pass: the only text that reaches embeddings and LLM calls'

In [ ]:
%%sql -r create_segments_stream
CREATE OR REPLACE STREAM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_STREAM
  ON DYNAMIC TABLE TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS SHOW_INITIAL_ROWS = TRUE
  COMMENT = 'New segments not yet redacted'

In [ ]:
%%sql -r create_redact_procedure
CREATE OR REPLACE PROCEDURE TRANSCRIPT_CHURN_ML_DEMO.CORE.REDACT_NEW_SEGMENTS()
RETURNS VARIANT LANGUAGE SQL
COMMENT = 'Refresh segments, AI_REDACT only the rows the stream has not seen, then remove any known staff name it missed'
EXECUTE AS CALLER
AS
$$
DECLARE
  n INT;
  fixes INT;
  pat STRING;
BEGIN
  ALTER DYNAMIC TABLE TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS REFRESH;
  -- Known people (here, the reps) are removed deterministically: AI_REDACT is best effort.
  SELECT '(' || LISTAGG(REP_NAME || '|' || SPLIT_PART(REP_NAME, ' ', 1), '|') || ')'
    INTO :pat FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.REPS;
  MERGE INTO TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_REDACTED t
  USING (SELECT s.*, AI_REDACT(s.SEGMENT_TEXT)::STRING AS R1, AI_REDACT(s.CUSTOMER_TEXT)::STRING AS R2
         FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_STREAM s WHERE METADATA$ACTION = 'INSERT') s
  ON t.SEGMENT_ID = s.SEGMENT_ID
  WHEN NOT MATCHED THEN INSERT (SEGMENT_ID, CALL_ID, ACCOUNT_ID, CALL_TS, CALL_TYPE, OPPORTUNITY_ID, SEGMENT_NO,
                                REDACTED_TEXT, REDACTED_CUSTOMER_TEXT, KNOWN_NAME_FIXES, REDACTED_AT)
    VALUES (s.SEGMENT_ID, s.CALL_ID, s.ACCOUNT_ID, s.CALL_TS, s.CALL_TYPE, s.OPPORTUNITY_ID, s.SEGMENT_NO,
            REGEXP_REPLACE(s.R1, :pat, '[NAME]'), REGEXP_REPLACE(s.R2, :pat, '[NAME]'),
            REGEXP_COUNT(s.R1, :pat) + REGEXP_COUNT(s.R2, :pat), CURRENT_TIMESTAMP()::TIMESTAMP_NTZ);
  n := SQLROWCOUNT;
  SELECT COUNT_IF(KNOWN_NAME_FIXES > 0) INTO :fixes FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_REDACTED;
  RETURN OBJECT_CONSTRUCT('redacted_rows', n, 'rows_needing_known_name_pass_total', fixes);
END;
$$

In [ ]:
%%sql -r redact_history
CALL TRANSCRIPT_CHURN_ML_DEMO.CORE.REDACT_NEW_SEGMENTS()

In [ ]:
%%sql -r redaction_before_after
SELECT s.SEGMENT_ID, LEFT(s.SEGMENT_TEXT, 220) AS RAW_TEXT, LEFT(r.REDACTED_TEXT, 220) AS REDACTED_TEXT
FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS s
JOIN TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_REDACTED r USING (SEGMENT_ID)
WHERE s.SEGMENT_TEXT ILIKE '%reach me at%' ORDER BY s.SEGMENT_ID LIMIT 3

In [ ]:
leak = q(f"""
SELECT COUNT(*) AS N,
       COUNT_IF(REGEXP_LIKE(REDACTED_TEXT, '.*[0-9]{{3}}-555-[0-9]{{4}}.*', 's')) AS PHONES_LEFT,
       COUNT_IF(REDACTED_TEXT ILIKE '%@homemail.example%') AS EMAILS_LEFT,
       COUNT_IF(EXISTS (SELECT 1 FROM {DB}.RAW.REPS r WHERE CONTAINS(REDACTED_TEXT, r.REP_NAME))) AS REP_NAMES_LEFT,
       COUNT_IF(REDACTED_TEXT LIKE '%[NAME]%') AS NAME_TAGS,
       COUNT_IF(REDACTED_TEXT LIKE '%[PHONE_NUMBER]%' OR REDACTED_TEXT LIKE '%[EMAIL]%') AS CONTACT_TAGS,
       COUNT_IF(KNOWN_NAME_FIXES > 0) AS KNOWN_NAME_FIX_ROWS
FROM {DB}.CORE.SEGMENTS_REDACTED""").iloc[0]
planted_phones = one(f"SELECT COUNT_IF(REGEXP_LIKE(SEGMENT_TEXT, '.*[0-9]{{3}}-555-[0-9]{{4}}.*', 's')) FROM {DB}.CORE.SEGMENTS")
check("S4", int(leak.N) == HIST_SEGMENTS and int(leak.PHONES_LEFT) == 0 and int(leak.EMAILS_LEFT) == 0
      and int(leak.REP_NAMES_LEFT) == 0 and planted_phones > 0,
      f"{int(leak.N):,} segments redacted; planted phone numbers {planted_phones} -> left {int(leak.PHONES_LEFT)}, "
      f"emails left {int(leak.EMAILS_LEFT)}, rep names left {int(leak.REP_NAMES_LEFT)}; "
      f"[NAME] tags {int(leak.NAME_TAGS)}, contact tags {int(leak.CONTACT_TAGS)}; "
      f"known-name pass caught {int(leak.KNOWN_NAME_FIX_ROWS)} rows AI_REDACT missed")

---
## 5. Embed redacted text only

`CORE.SEGMENT_EMBEDDINGS` reads `CORE.SEGMENTS_REDACTED`, never the raw column, and stores the embedding model name next to
each 1,024-dimension vector. If that model is retired, the column tells you which rows (and which topic exemplars) need a
planned re-embed. The table refreshes incrementally, so a new month embeds only new segments.

In [ ]:
%%sql -r create_embeddings
CREATE OR REPLACE DYNAMIC TABLE TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENT_EMBEDDINGS
  TARGET_LAG = 'DOWNSTREAM' WAREHOUSE = TRANSCRIPT_CHURN_ML_WH REFRESH_MODE = INCREMENTAL
  COMMENT = 'AI_EMBED of redacted segments only; EMBED_MODEL is stored so a model retirement means a planned re-embed'
AS
SELECT SEGMENT_ID, CALL_ID, ACCOUNT_ID, CALL_TS, CALL_TYPE, OPPORTUNITY_ID, REDACTED_TEXT, REDACTED_CUSTOMER_TEXT,
       'snowflake-arctic-embed-l-v2.0' AS EMBED_MODEL,
       AI_EMBED('snowflake-arctic-embed-l-v2.0', REDACTED_TEXT) AS EMBEDDING
FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENTS_REDACTED

In [ ]:
emb = q(f"""SELECT COUNT(*) AS N, COUNT(DISTINCT EMBED_MODEL) AS MODELS, MIN(VECTOR_L2_DISTANCE(EMBEDDING, EMBEDDING)) AS SELF_DIST
           FROM {DB}.CORE.SEGMENT_EMBEDDINGS""").iloc[0]
dt = session.sql(f"SHOW DYNAMIC TABLES LIKE 'SEGMENT_EMBEDDINGS' IN SCHEMA {DB}.CORE").collect()[0]
check("S5", int(emb.N) == HIST_SEGMENTS and dt["refresh_mode"] == "INCREMENTAL" and int(emb.MODELS) == 1
      and "SEGMENTS_REDACTED" in dt["text"] and "UTTERANCES" not in dt["text"],
      f"{int(emb.N):,} embeddings ({EMBED_MODEL}, 1024-d), refresh_mode {dt['refresh_mode']}; "
      f"the definition reads CORE.SEGMENTS_REDACTED only")

---
## 6. A new month lands

January's calls arrive the way the call-recording platform delivers them: rows appended to `RAW.CALLS` and
`RAW.UTTERANCES`. Running the same two steps again redacts and embeds only the January segments.

In [ ]:
%%sql -r load_new_month_calls
INSERT INTO TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS SELECT * FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.CALLS_NEW_MONTH

In [ ]:
%%sql -r load_new_month_turns
INSERT INTO TRANSCRIPT_CHURN_ML_DEMO.RAW.UTTERANCES SELECT * FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.UTTERANCES_NEW_MONTH

In [ ]:
t0 = time.time()
REDACT_NEW = json.loads(one(f"CALL {DB}.CORE.REDACT_NEW_SEGMENTS()"))["redacted_rows"]
refresh = session.sql(f"ALTER DYNAMIC TABLE {DB}.CORE.SEGMENT_EMBEDDINGS REFRESH").collect()[0]
EMBED_STATS = json.loads(refresh["statistics"]) if refresh["statistics"].startswith("{") else {}
NEW_MONTH_SECONDS = time.time() - t0
print(f"redacted {REDACT_NEW} new segments; embedding refresh statistics {refresh['statistics']} "
      f"({NEW_MONTH_SECONDS:.0f} s)")

In [ ]:
new_expected = one(f"SELECT COUNT(*) FROM {DB}.CORE.SEGMENTS WHERE CALL_TS >= '2026-01-01'")
total = one(f"SELECT COUNT(*) FROM {DB}.CORE.SEGMENT_EMBEDDINGS")
inserted = int(EMBED_STATS.get("insertedRows", -1))
NEW_SEGMENTS = new_expected
check("S6", REDACT_NEW == new_expected and inserted == new_expected and total == HIST_SEGMENTS + new_expected,
      f"January added {new_expected:,} segments: AI_REDACT ran on {REDACT_NEW:,} rows (history {HIST_SEGMENTS:,} untouched), "
      f"embedding refresh inserted {inserted:,}; total embeddings {total:,}")

---
## Summary

Raw text is masked, every segment is redacted before it is embedded, and each new month costs only its own rows.
Next: `02` assigns business topics and discovers the ones nobody defined yet.